# Run a project without Snowflake

`clair.run()` accepts an `adapter`. The adapter is the only part of clair that speaks to a
warehouse, and `WarehouseAdapter` is a small abstract class of eight methods. Thus you can
give clair an adapter that holds its tables in memory, and run the complete pipeline on
your machine.

This is useful for four things:

- a smoke test of a project in continuous integration, with no warehouse and no credits
- a look at the exact statements, in the exact order, that a run sends
- a demonstration of the staging and promotion mechanism
- a test of the behaviour after a Trouve fails

The adapter below records each statement. It executes no SQL, because a warehouse is the
thing that executes SQL. It does execute each pandas transform, because those run on this
machine in a real run too.

In [1]:
from pathlib import Path


def find_examples_root(start: Path | None = None) -> Path:
    """Give the directory that holds examples/projects.

    The search starts at the working directory and it goes up. Thus the
    notebook runs from this directory, from the repository root, or from a
    Jupyter server that you started at a different place.
    """
    for directory in [start or Path.cwd(), *(start or Path.cwd()).parents]:
        if (directory / "examples" / "projects").is_dir():
            return directory
    raise FileNotFoundError("No parent directory holds examples/projects.")


REPOSITORY_ROOT = find_examples_root()

# The CLI configures structlog; a notebook does not. This line keeps the INFO
# messages of each operation out of the cells. Remove it to read them.
import logging

import structlog

structlog.configure(wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING))

import os

import pandas as pd

import clair
from clair.adapters.base import QueryResult, WarehouseAdapter
from clair.trouves.address import TrouveAddress

PROJECT = REPOSITORY_ROOT / "examples" / "projects" / "example_4"

os.environ["CLAIR_USER"] = "notebook_user"

## An environment with no account

`clair.run()` reads the environment from `~/.clair/environments.yml`. This notebook must
not depend on your file, and it must not write to your home directory. Thus it makes an
environments file in a temporary directory, and it points clair at that file.

In [2]:
import tempfile
from pathlib import Path

import clair.environments.environments as environments_module

NOTEBOOK_DIRECTORY = Path(tempfile.mkdtemp(prefix="clair-notebook-"))
environments_file = NOTEBOOK_DIRECTORY / "environments.yml"
environments_file.write_text(
    """
dev:
  account: notebook
  user: notebook_user
  warehouse: notebook_warehouse
  role: NOTEBOOK
  password: unused-because-this-notebook-opens-no-connection
  threads: 1

prod:
  account: notebook
  user: notebook_user
  warehouse: notebook_warehouse
  role: NOTEBOOK
  password: unused-because-this-notebook-opens-no-connection
  threads: 4
""".strip()
)

# clair reads ~/.clair/environments.yml. This line points it at the file above.
environments_module.DEFAULT_ENVIRONMENTS_PATH = environments_file

print(environments_file.read_text())

dev:
  account: notebook
  user: notebook_user
  warehouse: notebook_warehouse
  role: NOTEBOOK
  password: unused-because-this-notebook-opens-no-connection
  threads: 1

prod:
  account: notebook
  user: notebook_user
  warehouse: notebook_warehouse
  role: NOTEBOOK
  password: unused-because-this-notebook-opens-no-connection
  threads: 4


## The adapter

Each method of `WarehouseAdapter` has one job:

| Method | Job |
|--------|-----|
| `connect` | open the connection |
| `new_connection` | make a second adapter for a second thread |
| `execute` | run one statement, and give the query ID and the row count |
| `set_context` | set the warehouse, the role and the database of the session |
| `table_exists` | tell clair if an incremental target is there |
| `fetch_dataframe` | read a table for a pandas Trouve |
| `write_dataframe` | write the result of a pandas Trouve, or a seed |
| `close` | close the connection |

A real adapter for a second warehouse — DuckDB, BigQuery, Postgres — implements the same
eight methods. The runner needs no change.

In [3]:
class InMemoryAdapter(WarehouseAdapter):
    """A complete adapter that holds its tables in memory.

    The adapter records each statement, and it gives a successful result. It
    parses no SQL: the caller names the statement text that must fail.

    Args:
        source_dataframes: The DataFrame of each address that a pandas Trouve
            reads. The key is the address text.
        fail_on: A statement that holds one of these texts gives a failure.
    """

    def __init__(self, source_dataframes=None, fail_on=()):
        self.source_dataframes = dict(source_dataframes or {})
        self.fail_on = tuple(fail_on)
        self.statements: list[str] = []
        self.tables: dict[str, pd.DataFrame] = dict(self.source_dataframes)
        self.query_number = 0

    def connect(self, profile: dict) -> None:
        self.profile = profile

    def new_connection(self) -> "InMemoryAdapter":
        return self

    def execute(self, sql: str) -> QueryResult:
        self.statements.append(sql)
        self.query_number += 1
        failure = next((text for text in self.fail_on if text in sql), None)
        # A data quality test query gives the rows that disobey the condition.
        # Thus a SELECT that gives zero rows is a test that passes.
        is_query = sql.lstrip().upper().startswith("SELECT")
        return QueryResult(
            query_id=f"notebook-query-{self.query_number:03d}",
            query_url="",
            success=failure is None,
            error=f"the notebook adapter fails a statement that holds '{failure}'" if failure else None,
            row_count=0 if failure or is_query else 42,
        )

    def set_context(self, warehouse=None, role=None, database_name=None) -> None:
        pass

    def table_exists(self, database_name: str, schema_name: str, table_name: str) -> bool:
        return f"{database_name}.{schema_name}.{table_name}" in self.tables

    def fetch_dataframe(self, address: TrouveAddress) -> pd.DataFrame:
        return self.tables[str(address)].copy()

    def write_dataframe(self, dataframe: pd.DataFrame, address: TrouveAddress) -> QueryResult:
        self.tables[str(address)] = dataframe.copy()
        self.query_number += 1
        return QueryResult(
            query_id=f"notebook-write-{self.query_number:03d}",
            query_url="",
            success=True,
            row_count=len(dataframe),
        )

    def close(self) -> None:
        pass


print(InMemoryAdapter.__doc__.splitlines()[0])

A complete adapter that holds its tables in memory.


## The data that the pandas Trouve reads

`example_4_database.derived.daily_event_counts` is a `PandasTrouve`. It reads the refined
events, thus the adapter must hold that table. The dev routing entry puts each Trouve in
the `notebook_user` database, and the run reads the staging address of the parent, so the
adapter gives the same DataFrame for each address that it does not hold.

In [4]:
refined_events = pd.DataFrame(
    {
        "event_id": ["1", "2", "3", "4", "5"],
        "user_id": ["usr_abc", "usr_abc", "usr_def", "usr_def", "usr_ghi"],
        "event_type": ["page_view", "button_click", "page_view", "form_submit", "purchase"],
        "event_date": pd.to_datetime(
            ["2024-01-15", "2024-01-15", "2024-01-15", "2024-01-15", "2024-01-15"]
        ).date,
    }
)


class NotebookAdapter(InMemoryAdapter):
    """An InMemoryAdapter that gives the sample events for an absent table."""

    def fetch_dataframe(self, address: TrouveAddress) -> pd.DataFrame:
        return self.tables.get(str(address), refined_events).copy()


refined_events

,event_id,user_id,event_type,event_date
0,1,usr_abc,page_view,2024-01-15
1,2,usr_abc,button_click,2024-01-15
2,3,usr_def,page_view,2024-01-15
3,4,usr_def,form_submit,2024-01-15
4,5,usr_ghi,purchase,2024-01-15


## The run

`clair.run()` takes the adapter and it closes no connection: the adapter belongs to you.
Each other argument is the same as `clair run` on the command line.

In [5]:
adapter = NotebookAdapter()

summary = clair.run(PROJECT, env="dev", adapter=adapter, threads=1)

print(f"run_id:    {summary.run_id}")
print(f"succeeded: {summary.succeeded_count}")
print(f"failed:    {summary.failed_count}")
print(f"skipped:   {summary.skipped_count}")

2026-08-28 18:49:34 [warning  ] run.no_account_locator         detail='Clair cannot show the query URLs.' env=dev


run_id:    01a04a906d4e75748dfaf40aff96e7a6
succeeded: 4
failed:    0
skipped:   0


In [6]:
pd.DataFrame(
    [
        {
            "trouve": result.logical_address.split(".", 1)[1],
            "status": result.status.value,
            "run mode": result.effective_run_mode.value if result.effective_run_mode else "",
            "statements": len(result.sql or []),
            "tests": len(result.test_results),
            "tests passed": sum(1 for test_result in result.test_results if test_result.passed),
            "rows": result.row_count,
        }
        for result in summary.results
    ]
)

,trouve,status,run mode,statements,tests,tests passed,rows
0,reference.event_type_labels,success,full_refresh,0,0,0,0
1,refined.events,success,full_refresh,1,0,0,42
2,derived.daily_event_counts,success,full_refresh,0,2,2,0
3,derived.top_event_types,success,full_refresh,1,0,0,42


`summary.render()` gives the text that the CLI prints. The objects above hold the same
data, thus a program reads the attributes.

## What the run sent, and in which order

The recorded statements show the staging mechanism. clair builds each Trouve at a
run-scoped staging address, it runs the data quality tests there, and it promotes the
object only after the tests pass. A reader of the production table never sees a table that
failed a test.

In [7]:
for index, statement in enumerate(adapter.statements, start=1):
    first_line = " ".join(statement.split())[:110]
    print(f"{index:>3}. {first_line}")

  1. CREATE DATABASE IF NOT EXISTS notebook_user
  2. CREATE SCHEMA IF NOT EXISTS notebook_user.reference
  3. CREATE DATABASE IF NOT EXISTS notebook_user
  4. CREATE SCHEMA IF NOT EXISTS notebook_user.refined
  5. CREATE DATABASE IF NOT EXISTS notebook_user
  6. CREATE SCHEMA IF NOT EXISTS notebook_user.derived
  7. -- staging: promote the tested table CREATE OR REPLACE TABLE notebook_user.reference.event_type_labels CLONE n
  8. -- staging: drop the staging object that clair promoted DROP TABLE IF EXISTS notebook_user.reference.event_typ
  9. CREATE OR REPLACE TABLE notebook_user.refined.events__clair_01a04a906d4e75748dfaf40aff96e7a6 AS ( select event
 10. -- staging: promote the tested table CREATE OR REPLACE TABLE notebook_user.refined.events CLONE notebook_user.
 11. -- staging: drop the staging object that clair promoted DROP TABLE IF EXISTS notebook_user.refined.events__cla
 12. SELECT event_date, event_type, COUNT(*) FROM notebook_user.derived.daily_event_counts__clair_01a04a90

In [8]:
staging_frame = pd.DataFrame(
    [
        {
            "trouve": result.logical_address.split(".", 1)[1],
            "built at (staging)": (result.staging_address or "").split(".", 1)[-1],
            "promoted to (physical)": result.physical_address,
        }
        for result in summary.results
    ]
)
staging_frame

,trouve,built at (staging),promoted to (physical)
0,reference.event_type_labels,reference.event_type_labels__clair_01a04a906d4...,notebook_user.reference.event_type_labels
1,refined.events,refined.events__clair_01a04a906d4e75748dfaf40a...,notebook_user.refined.events
2,derived.daily_event_counts,derived.daily_event_counts__clair_01a04a906d4e...,notebook_user.derived.daily_event_counts
3,derived.top_event_types,derived.top_event_types__clair_01a04a906d4e757...,notebook_user.derived.top_event_types


The physical address holds `notebook_user`, because the dev routing entry of this project
puts each Trouve in a database with the name of the person. The staging address holds the
run id, thus two runs never write to one staging table.

## The data quality tests

`test=True` is the default. clair runs the tests of a Trouve at the staging address, and
it promotes the object after each test passes.

In [9]:
pd.DataFrame(
    [
        {
            "trouve": test_result.physical_address.split(".", 1)[1],
            "test": test_result.test_type,
            "column": test_result.column_name or "(table)",
            "passed": test_result.passed,
            "failing rows": test_result.failing_row_count,
        }
        for test_result in summary.test_results
    ]
)

,trouve,test,column,passed,failing rows
0,derived.daily_event_counts,unique_columns,(table),True,0
1,derived.daily_event_counts,not_null,event_count,True,0


Each test passes here, because the adapter gives zero rows for a `SELECT`. A test that
gives one row or more marks the Trouve as a failure, and clair promotes no object: the
production table keeps the data of the run before. Change the adapter to give a row count
above zero for a `SELECT`, and you see that path.

## What happens after a Trouve fails?

Give the adapter a text that must fail. clair marks that Trouve as FAILURE, and it marks
each Trouve downstream as SKIPPED. The other branches of the DAG continue. `clair.run()`
raises no error, thus your notebook reads the summary.

In [10]:
failing_adapter = NotebookAdapter(fail_on=["refined"])
failed_summary = clair.run(PROJECT, env="dev", adapter=failing_adapter, threads=1)

for result in failed_summary.results:
    reason = ""
    if result.status.value == "failure":
        reason = result.error.splitlines()[0]
    elif result.status.value == "skipped":
        reason = f"waits for {result.skipped_by}"
    print(f"{result.status.value:<9} {result.logical_address:<48} {reason}")

2026-08-28 18:49:34 [warning  ] run.no_account_locator         detail='Clair cannot show the query URLs.' env=dev


2026-08-28 18:49:34 [warning  ] run.node.failure               duration_seconds=0.0 error="the notebook adapter fails a statement that holds 'refined'. Clair keeps the staging object at notebook_user.refined.events__clair_01a04a906eaf7e66a4677707a2774537, if the build made one" logical=example_4_database.refined.events physical=notebook_user.refined.events query_ids=['notebook-query-010']


success   example_4_database.reference.event_type_labels   
failure   example_4_database.refined.events                the notebook adapter fails a statement that holds 'refined'. Clair keeps the staging object at notebook_user.refined.events__clair_01a04a906eaf7e66a4677707a2774537, if the build made one
skipped   example_4_database.derived.daily_event_counts    waits for notebook_user.refined.events
skipped   example_4_database.derived.top_event_types       waits for notebook_user.refined.events


In [11]:
failed = failed_summary.failed[0]
print("the statement that failed:\n")
print(failed.sql[failed.failed_statement_index].strip()[:400])

the statement that failed:

CREATE OR REPLACE TABLE notebook_user.refined.events__clair_01a04a906eaf7e66a4677707a2774537 AS (
select
            event_id,
            user_id,
            event_type,
            occurred_at,
            occurred_at::date                   as event_date,

            -- page_view / button_click
            properties:page::string             as page,
            properties:referrer::string   


The seed succeeded, because it reads no other Trouve. The two Trouves below the failure
are SKIPPED, and `skipped_by` names the cause. clair continues each branch that the
failure does not touch.

## Use this in continuous integration

Two lines make a smoke test that needs no warehouse, no credits, and no secret:

```python
summary = clair.run(PROJECT, env="dev", adapter=NotebookAdapter(), threads=1)
assert summary.failed_count == 0, [result.error for result in summary.failed]
```

The test finds a reference to a Trouve that does not exist, a cycle in the DAG, a routing
entry that gives an invalid name, and a pandas transform that raises. It does not find a
SQL syntax fault, because no adapter here parses SQL. `clair compile` and a real
integration run find that.

## Next

- [04_author_trouves.ipynb](04_author_trouves.ipynb) — write a project in the notebook,
  and test a pandas transform with no warehouse.

In [12]:
import shutil

shutil.rmtree(NOTEBOOK_DIRECTORY)
clair.clean(PROJECT)
print("removed the temporary environment and the artifacts of this notebook")

removed the temporary environment and the artifacts of this notebook
